# 📓 MTG-Jamendo → parsed dataset (Google Colab + Drive)

Ολόκληρη η ροή με τα raw δεδομένα στο **Google Drive** και τα `.npz` να γράφονται σε **τοπικό δίσκο** (γρήγορα), με ασφαλές upload στο Drive μετά κάθε παρτίδα.

**Σειρά:** mount → config → deps → MTG metadata → `create_labels` → download audio → smoke test → supervisor → `create_labels --check` → final sync → (προαιρετικά) διαγραφή mp3.

⚠️ Μετά απενεργοποίηση runtime ξανατρέξε πρώτα το κελί **config**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
from pathlib import Path
import os, subprocess, sys

DRIVE_REPO = Path("/content/drive/MyDrive/music-analyzer_v2")
LOCAL_PARSED = Path("/content/parsed_dataset")
MTG_REPO = DRIVE_REPO / "mtg-jamendo-dataset"
DRIVE_PARSED = DRIVE_REPO / "parsed_dataset"

# αν το repo δεν υπάρχει ήδη στο Drive, κάνε clone
if not DRIVE_REPO.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/ChristosPaterakisntua/music-analyzer-v2.git",
         str(DRIVE_REPO)], check=True)

LOCAL_PARSED.mkdir(parents=True, exist_ok=True)

# environment που διαβάζει το music_analyzer_v2.utils (ΠΡΙΝ από κάθε import του πακέτου)
os.environ["MA2_RAW_DATASET"] = str(MTG_REPO)
os.environ["MA2_PARSED_DATASET"] = str(LOCAL_PARSED)
os.environ["MA2_CHECKPOINT_DIR"] = str(DRIVE_REPO / "checkpoints")
os.environ["MA2_SAVED_MODELS_DIR"] = str(DRIVE_REPO / "saved_models")
os.environ["PYTHONPATH"] = str(DRIVE_REPO / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, str(DRIVE_REPO / "src"))

print("repo     :", DRIVE_REPO)
print("raw audio:", MTG_REPO / "dataset")
print("npz dir  :", LOCAL_PARSED)
print("upload to:", DRIVE_PARSED)

repo     : \content\drive\MyDrive\music-analyzer_v2
raw audio: \content\drive\MyDrive\music-analyzer_v2\mtg-jamendo-dataset\dataset
npz dir  : \content\parsed_dataset
upload to: \content\drive\MyDrive\music-analyzer_v2\parsed_dataset


In [3]:
# torch είναι ήδη προεγκατεστημένο στον Colab — δεν το ξαναεγκαθιστούμε
%pip install -q librosa matplotlib pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


c:\Users\chris\Desktop\MyCode\music-analyzer_v2\.venv\Scripts\python.exe: No module named pip


In [4]:
# metadata της MTG (data/splits/split-0/*.tsv — μόνα MB).
# Ο downloader τα τραβάει από το επίσημο GitHub repo (δεν είναι στο audio CDN).
!{sys.executable} {DRIVE_REPO}/scripts/download_mtg_jamendo.py --mode splits --dest {MTG_REPO}


C:\Users\chris\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\python.exe: can't open file 'c:\\content\\drive\\MyDrive\\music-analyzer_v2\\scripts\\download_mtg_jamendo.py': [Errno 2] No such file or directory


In [5]:
# labels.json -> LOCAL_PARSED (χρησιμοποιείται ως manifest από το downloader)
completed = subprocess.run([sys.executable, "-m", "music_analyzer_v2.create_labels"])
assert completed.returncode == 0, "create_labels απέτυχε"
print("labels:", LOCAL_PARSED / "labels.json")

AssertionError: create_labels απέτυχε

In [ ]:
# τι θα κατέβει (subset από labels.json, quality=low).
# ΣΗΜΕΙΩΣΗ: πάντα από το universal raw_30s source:
#   https://cdn.freesound.org/mtg-jamendo/raw_30s/audio-low/<NN>/<id>.low.mp3
# Τα per-subset paths (autotagging_genre/..., autotagging_moodtheme/...) δίνουν 404
# για tracks που δεν ανήκουν στο εκείνο το subset.
!{sys.executable} {DRIVE_REPO}/scripts/download_mtg_jamendo.py --dry-run

In [ ]:
# δοκιμή: 10 αρχεία
!{sys.executable} {DRIVE_REPO}/scripts/download_mtg_jamendo.py --limit 10

In [ ]:
# ολόκληρο το subset (~33 GB, 8.457 tracks) στο Drive.
# Στο 300 Mbps: ~20-30 λεπτά. Διακόπηκε; Ξανατρέξε το — τα ολοκληρωμένα παραλείπονται,
# τα μισά κατεβασμένα συνεχίζουν (Range resume).
!{sys.executable} {DRIVE_REPO}/scripts/download_mtg_jamendo.py

In [ ]:
# smoke test: εξαγωγή 3 tracks για έλεγχο όλης της ροής (paths, npz, validation)
import music_analyzer_v2.create_dataset as create_dataset_module
create_dataset_module.create_parsed_dataset(max_tracks=3, num_workers=2)

In [ ]:
# watchdog: τρέχει create_dataset μέχρι να τελειώσει· μετά κάθε παρτίδα ανεβάζει
# τα νέα .npz στο Drive. Αν πεθάνει το session → ξανατρέξε το ίδιο κελί για συνέχιση.
completed = subprocess.run([
    sys.executable, str(DRIVE_REPO / "scripts" / "supervised_create_dataset.py"),
    "--upload-to", str(DRIVE_PARSED),
])
print("supervisor exit code:", completed.returncode)

In [ ]:
# τελικό labels.json: μόνο tracks με έγκυρο .npz (+ διαγραφή κακών npz)
completed = subprocess.run([sys.executable, "-m", "music_analyzer_v2.create_labels",
                            "--check", "--delete"])
assert completed.returncode == 0, "create_labels --check απέτυχε"

In [ ]:
# τελικό sweep: ανέβασμα labels.json + ό,τι απέμεινε και επαλήθευση απέναντι στο Drive
!{sys.executable} {DRIVE_REPO}/scripts/supervised_create_dataset.py --upload-only --upload-to {DRIVE_PARSED}

In [ ]:
# ΠΡΟΑΙΡΕΤΙΚΑ: ελευθέρωση ~45 GB στο Drive (σβήσιμο των parse-αρμένων mp3).
# Δοκίμασε πρώτα σε dry-run· μετά ξεσχόλιαρε το --confirm.
!{sys.executable} {DRIVE_REPO}/scripts/delete_parsed_mp3.py
# !{sys.executable} {DRIVE_REPO}/scripts/delete_parsed_mp3.py --confirm

In [ ]:
# ΝΕΑ ΣΥΝΕΔΡΙΑ (π.χ. για training): φέρνει τα .npz από το Drive πίσω σε τοπικό δίσκο
import shutil
LOCAL_PARSED.mkdir(parents=True, exist_ok=True)
restored = 0
for source in sorted(DRIVE_PARSED.glob("*.npz")):
    target = LOCAL_PARSED / source.name
    if target.exists() and target.stat().st_size == source.stat().st_size:
        continue
    shutil.copy2(source, target)
    restored += 1
labels = DRIVE_PARSED / "labels.json"
if labels.exists():
    shutil.copy2(labels, LOCAL_PARSED / "labels.json")
print(f"restored {restored} .npz -> {LOCAL_PARSED}")

### Σημειώσεις

- **Resume παντού:** downloader, create_dataset (skip όσα έχουν .npz) και supervisor συνεχίζουν από εκεί που σταμάτησαν.
- **Πηγή ήχου:** μόνο το universal `raw_30s` (`.../raw_30s/audio-low/<NN>/<id>.low.mp3`). Τα per-subset URLs (`autotagging_genre/...`, `autotagging_moodtheme/...`) επιστρέφουν **404** για tracks που δεν ανήκουν σε εκείνο το subset, ενώ το ίδιο αρχείο υπάρχει κανονικά στο `raw_30s`.
- **Μέγεθος:** το `labels.json` έχει 8.457 tracks (train 4.742 / validation 1.799 / test 1.916) — μόνο τα tracks που έχουν *και* genre *και* feeling στο target label set.
- **Χώρος Drive (~200 GB):** ~33 GB audio + τα .npz (~65 GB) + repo → άνετα. Μπορείς να σβήσεις τα mp3s αφού γίνει parse (`delete_parsed_mp3.py --confirm`).
- **Training:** τρέξε το `handle_training.ipynb` μετά το config κελί (MA2_PARSED_DATASET → τοπικό = γρήγορο I/O). Checkpoints ανεβαίνουν στο Drive μέσω `MA2_CHECKPOINT_DIR`· αν τα θέλεις τοπικά, αλλαξ΄το στο config.
- Ο παλιός C++ supervisor (`scripts/supervised_create_dataset.cpp`) μένει για Windows· εδώ χρησιμοποιείται ο Python (`supervised_create_dataset.py`).